# Module 5.2: Backtest Analysis

## Learning Objectives
By completing this notebook, you will:
1. Backtest LLM-generated trading signals on historical data
2. Calculate key performance metrics (Sharpe, win rate, profit factor)
3. Calibrate LLM confidence scores against actual outcomes
4. Optimize signal thresholds and position sizing

## Prerequisites
- Completion of Module 5.1 (Signal Generation)
- Understanding of backtesting concepts
- Familiarity with trading performance metrics

## Estimated Time: 90 minutes

---

## Setup & Imports

In [ ]:
# Purpose: Import libraries for backtesting
# Key Concept: Performance analysis and optimization tools

import os
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
np.random.seed(42)

print("✓ Imports complete")

## 1. Understanding Backtesting for LLM Signals

### Why Backtest LLM Signals?

LLM-generated signals have unique characteristics:

**Traditional Signals:**
- Deterministic rules (MA crossover, RSI threshold)
- Consistent execution
- Easy to backtest

**LLM Signals:**
- Probabilistic assessments
- Context-dependent
- Confidence scores need calibration
- Harder to backtest (requires historical LLM inference)

### Key Metrics

1. **Win Rate (Precision)**: % of profitable signals
2. **Profit Factor**: Gross profit / gross loss
3. **Sharpe Ratio**: (Return - RF) / Volatility
4. **Max Drawdown**: Largest peak-to-trough decline
5. **Confidence Calibration**: Does 70% confidence = 70% accuracy?

### Backtesting Approach

For this notebook, we'll:
1. Simulate historical signals (in production, you'd re-run LLM on historical data)
2. Generate synthetic price movements
3. Calculate P&L for each signal
4. Analyze performance and calibration

## 2. Simulating Historical Signals and Returns

In [ ]:
# Purpose: Generate synthetic historical signals for backtesting
# Key Concept: Realistic signal dataset with confidence scores

@dataclass
class HistoricalSignal:
    """Historical signal with outcome."""
    
    date: datetime
    commodity: str
    direction: str  # 'long' or 'short'
    confidence: float  # 0-1
    position_size: float
    entry_price: float
    exit_price: Optional[float] = None
    holding_days: Optional[int] = None
    pnl: Optional[float] = None
    return_pct: Optional[float] = None
    
    def calculate_pnl(self):
        """Calculate P&L and return."""
        if self.exit_price is None:
            return
        
        # Calculate price change
        price_change = self.exit_price - self.entry_price
        
        # Apply direction (long profits from price increase, short from decrease)
        if self.direction == 'short':
            price_change = -price_change
        
        # Calculate P&L and return
        self.return_pct = (price_change / self.entry_price)
        self.pnl = self.return_pct * self.position_size


def generate_historical_signals(num_signals: int = 100) -> List[HistoricalSignal]:
    """
    Generate synthetic historical signals with realistic characteristics.
    
    In production, this would:
    1. Load historical fundamental data
    2. Re-run LLM signal generation
    3. Match signals with actual price movements
    
    Parameters
    ----------
    num_signals : int
        Number of signals to generate
        
    Returns
    -------
    list of HistoricalSignal
        Simulated historical signals
    """
    signals = []
    
    start_date = datetime.now() - timedelta(days=365)
    commodities = ["WTI Crude", "Natural Gas", "RBOB Gasoline"]
    
    for i in range(num_signals):
        # Random signal characteristics
        date = start_date + timedelta(days=np.random.randint(0, 365))
        commodity = np.random.choice(commodities)
        direction = np.random.choice(['long', 'short'])
        
        # Confidence follows beta distribution (more signals at 60-80%)
        confidence = np.random.beta(5, 3)  # Peaks around 0.6-0.7
        
        # Position size based on confidence
        position_size = confidence * 100
        
        # Entry price
        if commodity == "WTI Crude":
            entry_price = np.random.uniform(60, 90)
        elif commodity == "Natural Gas":
            entry_price = np.random.uniform(2, 5)
        else:
            entry_price = np.random.uniform(2, 3.5)
        
        # Holding period (1-30 days)
        holding_days = np.random.randint(1, 31)
        
        # Generate return based on confidence (higher confidence = higher expected return)
        # Base return: normally distributed around confidence-adjusted mean
        mean_return = (confidence - 0.5) * 0.03  # -1.5% to +1.5% based on confidence
        volatility = 0.02  # 2% daily vol
        
        # Add skill edge: higher confidence signals actually perform better
        skill_edge = (confidence - 0.5) * 0.01
        actual_return = np.random.normal(mean_return + skill_edge, 
                                        volatility * np.sqrt(holding_days))
        
        # Calculate exit price
        exit_price = entry_price * (1 + actual_return)
        
        # Create signal
        signal = HistoricalSignal(
            date=date,
            commodity=commodity,
            direction=direction,
            confidence=confidence,
            position_size=position_size,
            entry_price=entry_price,
            exit_price=exit_price,
            holding_days=holding_days
        )
        
        signal.calculate_pnl()
        signals.append(signal)
    
    return signals


# Generate historical signals
historical_signals = generate_historical_signals(200)

print(f"Generated {len(historical_signals)} historical signals")
print(f"Date range: {min(s.date for s in historical_signals).date()} to {max(s.date for s in historical_signals).date()}")
print(f"Average confidence: {np.mean([s.confidence for s in historical_signals]):.2%}")

## 3. Performance Metrics Calculation

In [ ]:
# Purpose: Calculate comprehensive backtest metrics
# Key Concept: Quantifying signal quality

@dataclass
class BacktestMetrics:
    """Comprehensive backtest performance metrics."""
    
    # Basic metrics
    total_signals: int
    winning_signals: int
    losing_signals: int
    win_rate: float
    
    # P&L metrics
    total_pnl: float
    avg_win: float
    avg_loss: float
    profit_factor: float
    
    # Risk metrics
    sharpe_ratio: float
    max_drawdown: float
    max_drawdown_pct: float
    
    # Additional stats
    avg_holding_days: float
    total_return_pct: float
    
    def to_dict(self) -> Dict:
        """Convert to dictionary."""
        return {k: v for k, v in self.__dict__.items()}


class BacktestAnalyzer:
    """Analyze backtest performance."""
    
    def __init__(self, signals: List[HistoricalSignal]):
        self.signals = signals
        self.df = self._to_dataframe()
        
    def _to_dataframe(self) -> pd.DataFrame:
        """Convert signals to DataFrame."""
        data = []
        for signal in self.signals:
            data.append({
                'date': signal.date,
                'commodity': signal.commodity,
                'direction': signal.direction,
                'confidence': signal.confidence,
                'position_size': signal.position_size,
                'entry_price': signal.entry_price,
                'exit_price': signal.exit_price,
                'holding_days': signal.holding_days,
                'pnl': signal.pnl,
                'return_pct': signal.return_pct
            })
        return pd.DataFrame(data)
    
    def calculate_metrics(self) -> BacktestMetrics:
        """
        Calculate comprehensive backtest metrics.
        
        Returns
        -------
        BacktestMetrics
            Performance metrics
        """
        df = self.df
        
        # Basic counts
        total_signals = len(df)
        winning_signals = (df['pnl'] > 0).sum()
        losing_signals = (df['pnl'] <= 0).sum()
        win_rate = winning_signals / total_signals if total_signals > 0 else 0
        
        # P&L metrics
        total_pnl = df['pnl'].sum()
        avg_win = df[df['pnl'] > 0]['pnl'].mean() if winning_signals > 0 else 0
        avg_loss = abs(df[df['pnl'] <= 0]['pnl'].mean()) if losing_signals > 0 else 0
        
        gross_profit = df[df['pnl'] > 0]['pnl'].sum()
        gross_loss = abs(df[df['pnl'] <= 0]['pnl'].sum())
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
        
        # Risk metrics
        returns = df['return_pct'].values
        sharpe_ratio = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 0 else 0
        
        # Max drawdown
        cumulative_pnl = df.sort_values('date')['pnl'].cumsum()
        running_max = cumulative_pnl.expanding().max()
        drawdown = cumulative_pnl - running_max
        max_drawdown = drawdown.min()
        max_drawdown_pct = (max_drawdown / running_max.max() * 100) if running_max.max() > 0 else 0
        
        # Additional stats
        avg_holding_days = df['holding_days'].mean()
        initial_capital = 10000  # Assume $10k starting capital
        total_return_pct = (total_pnl / initial_capital) * 100
        
        return BacktestMetrics(
            total_signals=total_signals,
            winning_signals=winning_signals,
            losing_signals=losing_signals,
            win_rate=win_rate,
            total_pnl=total_pnl,
            avg_win=avg_win,
            avg_loss=avg_loss,
            profit_factor=profit_factor,
            sharpe_ratio=sharpe_ratio,
            max_drawdown=max_drawdown,
            max_drawdown_pct=max_drawdown_pct,
            avg_holding_days=avg_holding_days,
            total_return_pct=total_return_pct
        )


# Analyze performance
analyzer = BacktestAnalyzer(historical_signals)
metrics = analyzer.calculate_metrics()

print("=== BACKTEST PERFORMANCE ===")
print(f"\nBASIC METRICS:")
print(f"  Total Signals: {metrics.total_signals}")
print(f"  Winning Signals: {metrics.winning_signals}")
print(f"  Losing Signals: {metrics.losing_signals}")
print(f"  Win Rate: {metrics.win_rate:.2%}")

print(f"\nP&L METRICS:")
print(f"  Total P&L: ${metrics.total_pnl:.2f}")
print(f"  Average Win: ${metrics.avg_win:.2f}")
print(f"  Average Loss: ${metrics.avg_loss:.2f}")
print(f"  Profit Factor: {metrics.profit_factor:.2f}")

print(f"\nRISK METRICS:")
print(f"  Sharpe Ratio: {metrics.sharpe_ratio:.2f}")
print(f"  Max Drawdown: ${metrics.max_drawdown:.2f} ({metrics.max_drawdown_pct:.2f}%)")
print(f"  Total Return: {metrics.total_return_pct:.2f}%")

print(f"\nOTHER:")
print(f"  Avg Holding Days: {metrics.avg_holding_days:.1f}")

## 4. Confidence Calibration Analysis

Critical for LLM signals: Does the confidence score actually reflect probability of success?

In [ ]:
# Purpose: Analyze confidence calibration
# Key Concept: Validating LLM confidence scores

def analyze_confidence_calibration(signals: List[HistoricalSignal]) -> pd.DataFrame:
    """
    Analyze how well confidence scores predict outcomes.
    
    Parameters
    ----------
    signals : list of HistoricalSignal
        Historical signals with outcomes
        
    Returns
    -------
    pd.DataFrame
        Calibration analysis by confidence bucket
    """
    df = pd.DataFrame([{
        'confidence': s.confidence,
        'pnl': s.pnl,
        'is_winner': s.pnl > 0
    } for s in signals])
    
    # Create confidence buckets
    bins = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    labels = ['<50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']
    df['confidence_bucket'] = pd.cut(df['confidence'], bins=bins, labels=labels)
    
    # Calculate metrics by bucket
    calibration = df.groupby('confidence_bucket', observed=True).agg({
        'confidence': ['mean', 'count'],
        'is_winner': 'mean',
        'pnl': ['mean', 'sum']
    }).round(4)
    
    calibration.columns = ['avg_confidence', 'count', 'win_rate', 'avg_pnl', 'total_pnl']
    
    # Calculate calibration error
    calibration['calibration_error'] = abs(calibration['avg_confidence'] - calibration['win_rate'])
    
    return calibration


# Analyze calibration
calibration_df = analyze_confidence_calibration(historical_signals)

print("=== CONFIDENCE CALIBRATION ANALYSIS ===")
print(calibration_df)

# Overall calibration error
overall_error = calibration_df['calibration_error'].mean()
print(f"\nMean Absolute Calibration Error: {overall_error:.4f}")
print("\nInterpretation:")
if overall_error < 0.05:
    print("  ✓ EXCELLENT: Confidence scores are well-calibrated")
elif overall_error < 0.10:
    print("  ✓ GOOD: Confidence scores are reasonably calibrated")
else:
    print("  ✗ POOR: Confidence scores need recalibration")

In [ ]:
# Purpose: Visualize confidence calibration
# Key Concept: Calibration plot shows ideal vs actual

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Calibration curve
ax1 = axes[0]
confidence_midpoints = [0.55, 0.65, 0.75, 0.85, 0.95]
actual_win_rates = calibration_df['win_rate'].values

ax1.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration')
ax1.plot(calibration_df['avg_confidence'], actual_win_rates, 
         'o-', linewidth=2, markersize=10, color='blue', label='Actual')

# Add error bars
for i, (conf, win_rate) in enumerate(zip(calibration_df['avg_confidence'], actual_win_rates)):
    ax1.plot([conf, conf], [conf, win_rate], 'r--', alpha=0.5)
    
ax1.set_xlabel('Predicted Confidence', fontsize=12)
ax1.set_ylabel('Actual Win Rate', fontsize=12)
ax1.set_title('Confidence Calibration Curve', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0.5, 1.0)
ax1.set_ylim(0.5, 1.0)

# Plot 2: P&L by confidence bucket
ax2 = axes[1]
bucket_labels = calibration_df.index.astype(str)
avg_pnls = calibration_df['avg_pnl'].values
colors = ['green' if x > 0 else 'red' for x in avg_pnls]

bars = ax2.bar(bucket_labels, avg_pnls, color=colors, alpha=0.7)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)

# Add count labels on bars
for i, (bar, count) in enumerate(zip(bars, calibration_df['count'])):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'n={int(count)}', ha='center', va='bottom' if height > 0 else 'top',
            fontsize=9)

ax2.set_xlabel('Confidence Bucket', fontsize=12)
ax2.set_ylabel('Average P&L ($)', fontsize=12)
ax2.set_title('P&L by Confidence Level', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Performance by Signal Characteristics

In [ ]:
# Purpose: Analyze performance across different signal dimensions
# Key Concept: Identifying what makes signals successful

df = analyzer.df

# Performance by commodity
commodity_perf = df.groupby('commodity').agg({
    'pnl': ['count', 'sum', 'mean'],
    'return_pct': 'mean'
}).round(4)
commodity_perf.columns = ['count', 'total_pnl', 'avg_pnl', 'avg_return_pct']
commodity_perf['win_rate'] = df.groupby('commodity').apply(
    lambda x: (x['pnl'] > 0).mean()
).round(4)

print("=== PERFORMANCE BY COMMODITY ===")
print(commodity_perf)

# Performance by direction
direction_perf = df.groupby('direction').agg({
    'pnl': ['count', 'sum', 'mean'],
}).round(4)
direction_perf.columns = ['count', 'total_pnl', 'avg_pnl']
direction_perf['win_rate'] = df.groupby('direction').apply(
    lambda x: (x['pnl'] > 0).mean()
).round(4)

print("\n=== PERFORMANCE BY DIRECTION ===")
print(direction_perf)

# Performance by holding period
df['holding_bucket'] = pd.cut(df['holding_days'], 
                              bins=[0, 7, 14, 21, 31],
                              labels=['1-7 days', '8-14 days', '15-21 days', '22-30 days'])

holding_perf = df.groupby('holding_bucket', observed=True).agg({
    'pnl': ['count', 'mean'],
}).round(4)
holding_perf.columns = ['count', 'avg_pnl']
holding_perf['win_rate'] = df.groupby('holding_bucket', observed=True).apply(
    lambda x: (x['pnl'] > 0).mean()
).round(4)

print("\n=== PERFORMANCE BY HOLDING PERIOD ===")
print(holding_perf)

## 6. Equity Curve and Drawdown Analysis

In [ ]:
# Purpose: Visualize equity curve and drawdowns
# Key Concept: Understanding performance over time

# Sort by date and calculate cumulative P&L
df_sorted = df.sort_values('date').copy()
df_sorted['cumulative_pnl'] = df_sorted['pnl'].cumsum()
df_sorted['running_max'] = df_sorted['cumulative_pnl'].expanding().max()
df_sorted['drawdown'] = df_sorted['cumulative_pnl'] - df_sorted['running_max']

# Create visualizations
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Plot 1: Equity curve
ax1 = axes[0]
ax1.plot(df_sorted['date'], df_sorted['cumulative_pnl'], 
         linewidth=2, color='blue', label='Cumulative P&L')
ax1.fill_between(df_sorted['date'], 0, df_sorted['cumulative_pnl'],
                 where=df_sorted['cumulative_pnl']>=0, alpha=0.3, color='green')
ax1.fill_between(df_sorted['date'], 0, df_sorted['cumulative_pnl'],
                 where=df_sorted['cumulative_pnl']<0, alpha=0.3, color='red')
ax1.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax1.set_ylabel('Cumulative P&L ($)', fontsize=12)
ax1.set_title('Equity Curve', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Drawdown
ax2 = axes[1]
ax2.fill_between(df_sorted['date'], 0, df_sorted['drawdown'],
                 alpha=0.5, color='red', label='Drawdown')
ax2.set_ylabel('Drawdown ($)', fontsize=12)
ax2.set_title('Drawdown Over Time', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Rolling win rate
ax3 = axes[2]
window = 20
df_sorted['is_winner'] = (df_sorted['pnl'] > 0).astype(int)
df_sorted['rolling_win_rate'] = df_sorted['is_winner'].rolling(
    window=window, min_periods=1
).mean()

ax3.plot(df_sorted['date'], df_sorted['rolling_win_rate'] * 100,
         linewidth=2, color='purple', label=f'{window}-Signal Rolling Win Rate')
ax3.axhline(y=50, color='gray', linestyle='--', linewidth=1, label='50% (Random)')
ax3.set_xlabel('Date', fontsize=12)
ax3.set_ylabel('Win Rate (%)', fontsize=12)
ax3.set_title('Rolling Win Rate', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key statistics
final_pnl = df_sorted['cumulative_pnl'].iloc[-1]
max_equity = df_sorted['cumulative_pnl'].max()
max_dd = df_sorted['drawdown'].min()

print(f"\n=== EQUITY CURVE STATISTICS ===")
print(f"Final P&L: ${final_pnl:.2f}")
print(f"Peak Equity: ${max_equity:.2f}")
print(f"Max Drawdown: ${max_dd:.2f}")
print(f"Recovery: {'Yes' if final_pnl >= max_equity else 'No (still in drawdown)'}")

## 7. Threshold Optimization

Find the optimal minimum confidence threshold.

In [ ]:
# Purpose: Optimize confidence threshold for signal filtering
# Key Concept: Finding the minimum confidence that maximizes risk-adjusted returns

def evaluate_threshold(signals: List[HistoricalSignal], 
                      min_confidence: float) -> Dict:
    """
    Evaluate performance with a given confidence threshold.
    
    Parameters
    ----------
    signals : list of HistoricalSignal
        All historical signals
    min_confidence : float
        Minimum confidence threshold (0-1)
        
    Returns
    -------
    dict
        Performance metrics at this threshold
    """
    # Filter signals
    filtered = [s for s in signals if s.confidence >= min_confidence]
    
    if len(filtered) == 0:
        return {
            'threshold': min_confidence,
            'num_signals': 0,
            'win_rate': 0,
            'total_pnl': 0,
            'sharpe_ratio': 0,
            'profit_factor': 0
        }
    
    # Calculate metrics
    analyzer_thresh = BacktestAnalyzer(filtered)
    metrics_thresh = analyzer_thresh.calculate_metrics()
    
    return {
        'threshold': min_confidence,
        'num_signals': metrics_thresh.total_signals,
        'win_rate': metrics_thresh.win_rate,
        'total_pnl': metrics_thresh.total_pnl,
        'sharpe_ratio': metrics_thresh.sharpe_ratio,
        'profit_factor': metrics_thresh.profit_factor
    }


# Test different thresholds
thresholds = np.linspace(0.45, 0.85, 20)
threshold_results = [evaluate_threshold(historical_signals, t) for t in thresholds]
threshold_df = pd.DataFrame(threshold_results)

# Find optimal threshold (maximize Sharpe ratio)
optimal_idx = threshold_df['sharpe_ratio'].idxmax()
optimal_threshold = threshold_df.loc[optimal_idx, 'threshold']

print("=== THRESHOLD OPTIMIZATION ===")
print(f"\nOptimal Threshold: {optimal_threshold:.2%}")
print(f"Signals at Optimal: {threshold_df.loc[optimal_idx, 'num_signals']:.0f}")
print(f"Win Rate: {threshold_df.loc[optimal_idx, 'win_rate']:.2%}")
print(f"Sharpe Ratio: {threshold_df.loc[optimal_idx, 'sharpe_ratio']:.2f}")
print(f"Total P&L: ${threshold_df.loc[optimal_idx, 'total_pnl']:.2f}")

In [ ]:
# Purpose: Visualize threshold optimization
# Key Concept: Trade-off between signal count and quality

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Number of signals vs threshold
ax1 = axes[0, 0]
ax1.plot(threshold_df['threshold'] * 100, threshold_df['num_signals'],
         linewidth=2, marker='o', color='blue')
ax1.axvline(x=optimal_threshold * 100, color='red', linestyle='--',
           label=f'Optimal: {optimal_threshold:.2%}')
ax1.set_xlabel('Minimum Confidence (%)', fontsize=12)
ax1.set_ylabel('Number of Signals', fontsize=12)
ax1.set_title('Signal Count vs Threshold', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Win rate vs threshold
ax2 = axes[0, 1]
ax2.plot(threshold_df['threshold'] * 100, threshold_df['win_rate'] * 100,
         linewidth=2, marker='o', color='green')
ax2.axvline(x=optimal_threshold * 100, color='red', linestyle='--')
ax2.axhline(y=50, color='gray', linestyle=':', label='50% (Random)')
ax2.set_xlabel('Minimum Confidence (%)', fontsize=12)
ax2.set_ylabel('Win Rate (%)', fontsize=12)
ax2.set_title('Win Rate vs Threshold', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Sharpe ratio vs threshold
ax3 = axes[1, 0]
ax3.plot(threshold_df['threshold'] * 100, threshold_df['sharpe_ratio'],
         linewidth=2, marker='o', color='purple')
ax3.axvline(x=optimal_threshold * 100, color='red', linestyle='--')
ax3.scatter([optimal_threshold * 100], 
           [threshold_df.loc[optimal_idx, 'sharpe_ratio']],
           s=200, c='red', marker='*', zorder=5, label='Optimal')
ax3.set_xlabel('Minimum Confidence (%)', fontsize=12)
ax3.set_ylabel('Sharpe Ratio', fontsize=12)
ax3.set_title('Sharpe Ratio vs Threshold (Optimization Target)', 
             fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Total P&L vs threshold
ax4 = axes[1, 1]
ax4.plot(threshold_df['threshold'] * 100, threshold_df['total_pnl'],
         linewidth=2, marker='o', color='orange')
ax4.axvline(x=optimal_threshold * 100, color='red', linestyle='--')
ax4.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax4.set_xlabel('Minimum Confidence (%)', fontsize=12)
ax4.set_ylabel('Total P&L ($)', fontsize=12)
ax4.set_title('Total P&L vs Threshold', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise 5.2: Optimize Your Signal Strategy

In [ ]:
# Exercise: Optimize signal strategy
#
# Task:
# 1. Filter signals to only trade one commodity (your choice)
# 2. Find the optimal confidence threshold for that commodity
# 3. Calculate the improvement in Sharpe ratio
# 4. Visualize the optimized equity curve

# YOUR CODE HERE
# ---------------

# Step 1: Filter to one commodity
chosen_commodity = "WTI Crude"  # Change this
commodity_signals = None  # Filter historical_signals

# Step 2: Find optimal threshold
optimal_commodity_threshold = None

# Step 3: Calculate improvement
baseline_sharpe = None
optimized_sharpe = None
improvement = None

## Summary

### Key Takeaways

1. **Comprehensive Metrics**: Evaluate signals using win rate, profit factor, Sharpe ratio, and max drawdown
2. **Confidence Calibration**: LLM confidence scores must be validated against actual outcomes
3. **Threshold Optimization**: Finding the right minimum confidence balances signal count and quality
4. **Equity Curves**: Visualizing cumulative P&L reveals strategy robustness
5. **Segmentation Analysis**: Performance varies by commodity, direction, and holding period

### Module 5 Complete

You've now built:
- **5.1**: Signal generation pipeline with confidence-based sizing
- **5.2**: Comprehensive backtesting and optimization framework

### What's Next

In **Module 6: Production Systems**, we'll:
- Deploy signal pipelines to production
- Implement monitoring and alerting
- Build data ingestion workflows
- Handle errors and edge cases gracefully

### Additional Resources

- "Advances in Financial Machine Learning" by Marcos López de Prado
- "Quantitative Trading" by Ernie Chan
- Calibration: Niculescu-Mizil & Caruana (2005)
- Backtesting best practices: QuantStart, QuantConnect

---

**Practice Exercise**: Re-run this backtest monthly with actual signals from your live pipeline. Track how backtest metrics correlate with forward performance.